# 11 外观自定义：不用改代码也能调整网页 UI

## 1. 本节点目标
给页面管理员增加统一的“外观设计”窗口。用户可直接选择页面大背景和不同区域，设置颜色、渐变、透明度与毛玻璃，不需要打开 Python 文件。

## 2. 完成结果与验收
- “外观设计”窗口只有一套连续的区域颜色编辑器，不再拆成两个标签。
- 24 个区域覆盖页面大背景、顶部、导航、内容区、四类卡片、图片、分类标签、小状态框、记录框和按钮。
- 四类物品的浅色背景、边框和深色强调会由一个主题色自动生成。
- 保存后写入 SQLite，重新登录仍能恢复。
- 支持一键恢复默认主题。
- 自动测试总计 70 项通过。

## 3. 本节点文件结构
- `app.py`：显示合并后的外观设计窗口、生成预览并应用 CSS。
- `src/smart_laundry/accounts.py`：定义外观数据并负责读取、校验、保存和重置。
- `src/smart_laundry/database.py`：创建 `appearance_preferences` 表。
- `tests/test_accounts.py`：验证设置可以持久保存、恢复默认并拒绝非法值。

## 4. 关键代码解释
`AppearancePreferences` 是一张外观设置清单。`get_appearance_preferences()` 在用户还没有保存设置时返回默认值；`update_appearance_preferences()` 先检查颜色和数值范围，再写入数据库。

`_mix_with_white()` 将用户选择的一个深色与白色混合，自动得到卡片浅背景和中等深度边框，因此页面能保持清楚的深浅层次。

In [ ]:
def mix_with_white(rgb, white_ratio):
    return tuple(round(channel + (255 - channel) * white_ratio) for channel in rgb)

mix_with_white((57, 157, 140), 0.8)  # 深薄荷色变为浅卡片底色

## 5. 数据流
1. 页面管理员登录并点击“外观设计”。
2. 页面从 SQLite 读取该账户的偏好。
3. 用户调整控件并查看预览。
4. 点击保存后，输入经过校验并写入数据库。
5. Streamlit 重新运行页面，把偏好转换成 CSS 和物品卡片渐变。

## 6. 关键概念
- **主题变量**：集中保存页面与分类颜色等视觉参数。
- **CSS 覆盖层**：在基础样式之后应用账户自己的选择。
- **持久化**：设置写进 SQLite，程序关闭后不会丢失。
- **输入校验**：只接受安全的十六进制颜色和规定范围内的数值。

## 7. 为什么这样设计
没有引入复杂的前端框架，而是继续使用 Streamlit 原生控件。布局由程序固定，页面管理员发布的配色供所有用户查看；共享物品数据不会被主题影响。

## 8. 常见错误与排查
- **保存后没有变化**：先确认点击了“保存并应用”，再刷新页面。
- **颜色太接近看不清**：提高文字色与背景色的明暗差。
- **一页卡片太挤**：当前固定每页最多五件，并由响应式布局自动调整。
- **想撤销所有调整**：点击“恢复默认主题”。

## 9. 面试可能追问
**问：为什么不把设置只放在 session state？**
答：session state 在服务重启或重新登录后可能丢失，SQLite 才能提供真正的持久化。

**追问：怎样避免用户写入危险 CSS？**
答：用户不能输入任意 CSS，只能使用受控选择器；颜色还会经过正则校验，数值都有范围限制。

## 10. 必须掌握的最少知识
理解“选择控件产生值 → 校验 → 写入数据库 → 转成 CSS → 页面重新渲染”这条链路即可，不需要背全部 CSS 属性。

## 11. 可自测小题
1. 为什么每个账户可以使用不同主题？
2. 为什么一个分类颜色能生成多种深浅？
3. 为什么不再开放每页数量和间距调整？

<details><summary>参考答案</summary>1. 只有页面管理员能保存全站主题。2. 程序按比例与白色混合。3. 固定五卡布局能避免挤压、错位和底边不齐。</details>

## 12. 动手小练习
1. 保存一套深蓝主色、灰粉提醒色的主题，再恢复默认。
2. 分别调整四类物品颜色，观察程序自动生成的浅背景和边框。

## 13. 本节点术语表
- Theme：主题。
- CSS：控制网页视觉样式的规则。
- Hex Color：形如 `#7777DA` 的十六进制颜色。
- Preview：保存前看到的预览效果。

## 14. 下一节点连接
以后可以在此基础上增加“主题方案名称”和导入/导出，让用户保存多套主题并在家庭成员之间分享，但不会影响现有业务数据。